In [44]:
import pandas as pd
import requests
import json

# rapidApi headers
headers = {
    "X-RapidAPI-Key": "REDACTED_RAPIDAPI_KEY",
    "X-RapidAPI-Host": "api-nba-v1.p.rapidapi.com"
}

#########################################################
# team code by id
url = "https://api-nba-v1.p.rapidapi.com/teams"
response = requests.get(url, headers=headers).json()['response']
df = pd.DataFrame(response)
df = df.loc[(df['nbaFranchise'] == True) & (df['allStar'] == False)]
nba_teams_df = df[['id', 'code']]
#########################################################

In [45]:
# display all nba teams with code
display(nba_teams_df)

,id,code
0,1,ATL
1,2,BOS
3,4,BKN
4,5,CHA
5,6,CHI
6,7,CLE
7,8,DAL
8,9,DEN
9,10,DET
10,11,GSW


In [46]:
#########################################################
# get standings for team df
import time
def get_standings_df(season, team):
    url = "https://api-nba-v1.p.rapidapi.com/standings"
    querystring = {"league":"standard","season":season,"team":team}
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()['response'][0]
    )

    total_games = df.iloc[0]['win.total'] + df.iloc[0]['loss.total']
    wins_df = pd.DataFrame(
        [
            [
                team, 
                df.iloc[0]['win.home'], 
                df.iloc[0]['win.total'], 
                total_games,
                df.iloc[0]['win.home'] / df.iloc[0]['win.total'],
                df.iloc[0]['win.home'] / total_games,
                df.iloc[0]['win.total'] / total_games,
            ]
        ], 
        columns=[
            "team_id", 
            "win_home", 
            "win_total", 
            "total_games",
            "win_home_/_win_total",
            "win_home_/_total_games",
            "win_total_/_total_games",
        ]
    )
    return wins_df
#########################################################

In [47]:
# display nba standings win percentages
nba_standings_df = nba_teams_df.apply(lambda x: get_standings_df("2023", str(x["id"])), axis=1).array
nba_standings_df = pd.concat(nba_standings_df)
display(nba_standings_df)

,team_id,win_home,win_total,total_games,win_home_/_win_total,win_home_/_total_games,win_total_/_total_games
0,1,16,29,63,0.551724,0.253968,0.460317
0,2,29,48,62,0.604167,0.467742,0.774194
0,4,16,25,63,0.640000,0.253968,0.396825
0,5,8,15,63,0.533333,0.126984,0.238095
0,6,16,31,63,0.516129,0.253968,0.492063
0,7,22,41,63,0.536585,0.349206,0.650794
0,8,19,35,63,0.542857,0.301587,0.555556
0,9,25,43,63,0.581395,0.396825,0.682540
0,10,5,10,62,0.500000,0.080645,0.161290
0,11,17,33,62,0.515152,0.274194,0.532258


In [48]:
#########################################################
# games by game ids #####################################
def get_games_by_game_ids(season, team):
    url = "https://api-nba-v1.p.rapidapi.com/games"
    querystring = {"season":season,"team":team}
    headers = {
        "X-RapidAPI-Key": "REDACTED_RAPIDAPI_KEY",
        "X-RapidAPI-Host": "api-nba-v1.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    df = df.loc[(df["status.long"] == "Finished")]
    df = df.sort_values(by=["date.start"])
    
    # adding win col to df
    df['win'] = ''
    
    df.loc[
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = True
    
    df.loc[
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = False
    
    return df
#########################################################

In [80]:
cols_to_drop_for_player_stats = [
    'comment',
    'player.firstname',
    'player.lastname',
    'player.id',
    'team.id',
    'team.nickname',
    'team.code',
    'team.name',
    'team.logo',
    'game.id'
]

cols_to_drop_for_game_stats = [
    'league',
    'season',
    'stage',
    'officials',
    'timesTied',
    'leadChanges',
    'nugget',
    'date.start',
    'date.end',
    'date.duration',
    'status.clock',
    'status.halftime',
    'status.short',
    'status.long',
    'periods.current',
    'periods.total',
    'periods.endOfPeriod',
    'arena.name',
    'arena.city',
    'arena.state',
    'arena.country',
    'teams.visitors.id',
    'teams.visitors.name',
    'teams.visitors.nickname',
    'teams.visitors.code',
    'teams.visitors.logo',
    'teams.home.id',
    'teams.home.name',
    'teams.home.nickname',
    'teams.home.code',
    'teams.home.logo',
    'scores.visitors.win',
    'scores.visitors.loss',
    'scores.visitors.series.win',
    'scores.visitors.series.loss',
    'scores.visitors.linescore',
    'scores.home.win',
    'scores.home.loss',
    'scores.home.series.win',
    'scores.home.series.loss',
    'scores.home.linescore'
]

In [81]:
#########################################################
# drop all cols from a df ##############################
def drop_cols(df, cols):
    for col in cols:
        df = df.drop(col, axis=1)
    return df
#########################################################

In [82]:
#########################################################
# get player stats by game by
num_mapping_table = str.maketrans({'-': '', '.': '', '+': ''})

def get_top_players_per_game_df(n, team, game_id):    
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"

    querystring = {"game": game_id }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )

    df_opponent = df.loc[df['team.id'] != team]
    df = df.loc[df['team.id'] == team]
    
    df_opponent = drop_cols(df_opponent, cols_to_drop_for_player_stats)
    df = drop_cols(df, cols_to_drop_for_player_stats)

    df_opponent["plusMinus"] = df["plusMinus"].apply(lambda x: 
        float(x) if x.translate(num_mapping_table).isnumeric() else 0.0)
    df_opponent = df_opponent.add_prefix("opponent.")
    
    df["plusMinus"] = df["plusMinus"].apply(lambda x: 
        float(x) if x.translate(num_mapping_table).isnumeric() else 0.0)
    return { 
        "opponent": df_opponent.nlargest(n, "opponent.plusMinus"), 
        "friendly": df.nlargest(n, "plusMinus") 
    }
#########################################################

In [83]:
#########################################################
# flatten data frames ####################
def flatten_df(df):
    # Flatten the DataFrame
    flattened_data = {}
    for col in df.columns:
        for row in range(df.shape[0]):
            new_col_name = f"{col}{row}"
            flattened_data[new_col_name] = df[col].iloc[row]

    # Convert to DataFrame
    return pd.DataFrame([flattened_data])
#########################################################

In [84]:
#########################################################
# per team, per game, get top 5 players by plusMinus metric
def get_top_five_players_per_game_per_team_id(team_id_list, games_by_game_ids_by_team_ids): 
    top_five_players_per_game_per_team_id = {}

    for team_id in team_id_list:
        top_5_players_on_team_per_game = {}
        for game_id in games_by_game_ids_by_team_ids.get(team_id)['id'].array:
            # transform player_stats_df
            top_five_players_stats = get_top_players_per_game_df(5, team_id, game_id)
            opponent_top_five_player_stats = flatten_df(top_five_players_stats.get("opponent"))
            friendly_top_five_player_stats = flatten_df(top_five_players_stats.get("friendly"))
            top_five_players_stats = pd.concat(
                [opponent_top_five_player_stats, friendly_top_five_player_stats], 
                axis=1
            )
            ###########################
            top_5_players_on_team_per_game[game_id] = top_five_players_stats
            
        top_five_players_per_game_per_team_id[team_id] = top_5_players_on_team_per_game
    return top_five_players_per_game_per_team_id
#########################################################

In [85]:
#########################################################
# create training and test data
#########################################################

In [86]:
# Ex: LAKERS
lakers_team_id = nba_teams_df.loc[(df["code"] == "LAL")].iloc[0]["id"]

games_by_game_ids_by_team_ids = {}
lakers_games = get_games_by_game_ids("2023", lakers_team_id) 
lakers_games = drop_cols(lakers_games, cols_to_drop_for_game_stats)

games_by_game_ids_by_team_ids[lakers_team_id] = lakers_games
    
lakers_player_stats_per_game = get_top_five_players_per_game_per_team_id(
    [lakers_team_id], 
    games_by_game_ids_by_team_ids
).get(lakers_team_id)

In [ ]:
def get_win_prc(game_df):
    total_win_prc = []
    last_ten_win_prc = []
    last_ten_games_win_loss = []
    last_ten_win_count = 0
    total_win_count = 0
    total_game_count = 0
    
    for index, row in game_df.iterrows():
        total_game_count += 1
        last_ten_games_win_loss.append(row['win'])
        
        if row['win']:
            total_win_count += 1
            last_ten_win_count += 1
            
        last_ten_win_prc.append(total_win_count)
        total_win_prc.append()
        
        if total_game_count > 10:
            if last_ten_games_win_loss.pop(0):
                last_ten_win_count -= 1
        else:
            las

In [90]:
display(lakers_player_stats_per_game.get(12479))
display(games_by_game_ids_by_team_ids.get(lakers_team_id))



,opponent.points0,opponent.points1,opponent.points2,opponent.points3,opponent.points4,opponent.pos0,opponent.pos1,opponent.pos2,opponent.pos3,opponent.pos4,...,blocks0,blocks1,blocks2,blocks3,blocks4,plusMinus0,plusMinus1,plusMinus2,plusMinus3,plusMinus4
0,4,6,6,8,10,F,SF,PG,PG,SG,...,1,0,1,0,0,8.0,7.0,6.0,2.0,-4.0


,id,scores.visitors.points,scores.home.points,win
0,12479,108.0,125.0,False
1,12488,126.0,129.0,True
2,12499,101.0,109.0,True
3,12509,129.0,125.0,False
4,12516,108.0,97.0,False
...,...,...,...,...
64,13404,131.0,134.0,True
65,13417,124.0,114.0,False
66,13432,104.0,116.0,True
67,13449,130.0,120.0,False
